In [4]:
import fastf1
import pandas as pd
import json

from F1DB import connect, query, execute, create_table, create_view, drop_table, drop_view, delete, run_sql_file, interactive
connect()

In [ ]:
SEASONS = [2022, 2023, 2024, 2025, 2026]

def get_season_events(
    season: int 
    ) -> pd.DataFrame:
    """Get all the events for the season

    Args:
        season (int): Season year (e.g. 2022)

    Returns:
        pd.DataFrame: Dataframe
    """
    season_events = fastf1.get_event_schedule(season)
    season_events = season_events[season_events['RoundNumber'] != 0]  # Remove testing events and only keep OfficialEventName, EventName, RoundNumber, EventDate, EventFormat
    season_events = season_events.reset_index(drop=True)
    # add season column
    season_events['Season'] = season
    # reorder columns
    season_events = season_events[['Season', 'OfficialEventName', 'EventName', 'RoundNumber', 'EventDate', 'EventFormat', 'Location', 'Country', 'Session1', 'Session2', 'Session3', 'Session4', 'Session5', 'Session1Date', 'Session2Date', 'Session3Date', 'Session4Date', 'Session5Date', 'Session1DateUtc', 'Session2DateUtc', 'Session3DateUtc', 'Session4DateUtc', 'Session5DateUtc']]
    return season_events

overall_season_events = pd.DataFrame()
for SEASON in SEASONS:
    season_events = get_season_events(SEASON)
    overall_season_events = pd.concat([overall_season_events, season_events], ignore_index=True)

connect()
create_table("season_events", overall_season_events)

In [5]:
season_events = query("SELECT * FROM season_events")

In [11]:
def get_event_sessions(season: int, round_number: int) -> pd.DataFrame:
    """Get all the sessions for the event

    Args:
        season (int): Season year (e.g. 2022)
        round_number (int): Round number (e.g. 1 for the first event of the season)

    Returns:
        pd.DataFrame: Dataframe with session information
    """
    event = fastf1.get_event(season, round_number)
    sessions_data = {}
    identifiers = ['FP1','FP2','FP3','Q','R','S','SQ','SS']
    for ident in identifiers:
        try:
            session_name = event.get_session_name(ident)
            session_date = event.get_session_date(ident, utc=True)
            session = event.get_session(ident)
            session.load()
            sessions_data[ident] = {
                "session_key": session_name,
                "date_utc": str(session_date),
                "session_object": session
            }
        except Exception:
            continue
    return sessions_data

season = season_events.loc[0, 'Season']
round_number = season_events.loc[0, 'RoundNumber']
event_sessions = get_event_sessions(season, round_number)
print(event_sessions)

core           INFO 	Loading data for Bahrain Grand Prix - Practice 1 [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 

{'FP1': {'session_key': 'Practice 1', 'date_utc': '2022-03-18 12:00:00', 'session_object': 2022 Season Round 1: Bahrain Grand Prix - Practice 1}, 'FP2': {'session_key': 'Practice 2', 'date_utc': '2022-03-18 15:00:00', 'session_object': 2022 Season Round 1: Bahrain Grand Prix - Practice 2}, 'FP3': {'session_key': 'Practice 3', 'date_utc': '2022-03-19 12:00:00', 'session_object': 2022 Season Round 1: Bahrain Grand Prix - Practice 3}, 'Q': {'session_key': 'Qualifying', 'date_utc': '2022-03-19 15:00:00', 'session_object': 2022 Season Round 1: Bahrain Grand Prix - Qualifying}, 'R': {'session_key': 'Race', 'date_utc': '2022-03-20 15:00:00', 'session_object': 2022 Season Round 1: Bahrain Grand Prix - Race}}


In [ ]:
# event_sessions['R']['session_object'].laps # add Season, RoundNumber EventName and SessionKey to laps dataframe and store in database
# event_sessions['R']['session_object'].results # add Season, RoundNumber EventName and SessionKey to results dataframe and store in database
# event_sessions['R']['session_object'].car_data # add Season, RoundNumber EventName and SessionKey, Driver Name, Car Number to internal dataframes
# event_sessions['R']['session_object'].pos_data # add Season, RoundNumber EventName and SessionKey, Driver Name, Car Number to internal dataframes
event_sessions['R']['session_object'].t0_date # add Season, RoundNumber EventName and SessionKey to telemetry dataframe and store in database


14

In [ ]:
def get_session_data(session) -> pd.DataFrame:
    session.load()
    results = session.results
    laps = session.laps
    telemetry = sessio